# Shared source and feature preflight (EXT-S0)

**Purpose:** Check the common prepared source, canonical gait features and fixed participant assignments for E3.4 and E5.3.

**Required inputs:** Prepared Baček HDF5; EXT_S0_Original_Reference_Records.zip.

**Outputs:** Canonical feature records and verification ZIP.

**Usage:** Run before E3.4 and E5.3. Files are read from the notebook’s current working directory unless specified in its configuration.


In [ ]:
# 1 — Fixed paths, release identity and immutable source checks
from pathlib import Path
import csv, hashlib, io, json, platform, sys, zipfile
import h5py
import numpy as np

H5 = Path('bacek_clean_common_angles_metabolic_dataset.h5')
REFERENCE = Path('EXT_S0_Original_Reference_Records.zip')
OUT = Path('EXT_S0_Shared_Preflight_results')
AMENDMENT_ID = 'FYP-JA-2026-11-04-02'
EXPECTED_SOURCE_SHA = 'c73ce561e94de52f1dc86128829c43ef536c4cf6b5f4aa9af1ea060409221fa3'
EXPECTED_REFERENCE_SHA = '5fd8b80d99522f82c3d2f45000dc553edf9afe5b125ea862d7da9bb89dba33ce'
EXPECTED_BOOTSTRAP_SHA = '5d93bf4ef62f3d488243f0b1732c8260fde78e8e1e1dba7afe6fb79390660373'
CHANNELS = ('hip_left', 'hip_right', 'knee_left', 'knee_right',
            'ankle_left', 'ankle_right', 'pelvis_sagittal',
            'pelvis_frontal', 'pelvis_transverse')
SPEEDS = (1.1, 0.8, 0.4)
CADENCES = (0.90, 0.95, 1.00, 1.10, 1.20)
OUTER = {
 1: ('S08','S10','S16','S21'),
 2: ('S11','S13','S17','S20'),
 3: ('S01','S02','S05','S18'),
 4: ('S03','S07','S12','S14','S19'),
 5: ('S04','S06','S09','S15'),
}
FOLD_OF = {p: f for f, members in OUTER.items() for p in members}
ATOL_DEGREES = 1e-10


def check(condition, message):
    if not bool(condition):
        raise AssertionError(message)


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def csv_rows(blob):
    return list(csv.DictReader(io.StringIO(blob.decode('utf-8'))))


def split_people(value):
    return set(x for x in value.split(';') if x)

check(H5.is_file(), f'Missing corrected Baček file: {H5}')
check(REFERENCE.is_file(), f'Missing archived reference package: {REFERENCE}')
check(sha256_file(REFERENCE) == EXPECTED_REFERENCE_SHA, 'Reference package hash mismatch')
source_sha = sha256_file(H5)
check(source_sha == EXPECTED_SOURCE_SHA, 'Corrected Baček HDF5 SHA-256 mismatch; STOP')
with zipfile.ZipFile(REFERENCE) as z:
    baseline = json.loads(z.read('04/run_summary.json'))
    ref_ledger = csv_rows(z.read('04/eligible_trial_ledger.csv'))
    ref_outer = csv_rows(z.read('04/outer_fold_manifest.csv'))
    ref_inner = csv_rows(z.read('04/inner_fold_manifest.csv'))
    ref_checks = csv_rows(z.read('04/verification_checks.csv'))
    bootstrap_blob = z.read('14/bootstrap_resampled_participants.csv')
check(hashlib.sha256(bootstrap_blob).hexdigest() == EXPECTED_BOOTSTRAP_SHA,
      'Archived E5.2 population bootstrap checksum mismatch')
check(baseline['status'] == 'PASS' and baseline['source_sha256'] == source_sha,
      'Original foundational run identity or PASS state mismatch')
check(len(ref_checks) > 0 and all(r['status'] == 'PASS' for r in ref_checks),
      'Original foundational verification records not all PASS')
print('Verified approved source, original model-reference archive and bootstrap checksum.')

## Synthetic numerical checks


In [ ]:
# 2 — Two independent numerical implementations and analytic synthetic fixtures

def frozen_means(channels):
    return np.stack([np.asarray(channels[name], dtype=np.float64).mean(axis=0, dtype=np.float64)
                     for name in CHANNELS])


def independent_means(channels):
    return np.stack([np.sum(np.asarray(channels[name], dtype=np.float64), axis=0,
                            dtype=np.float64) / len(channels[name]) for name in CHANNELS])


def frozen_d(means):
    values = []
    for line in means:
        for q in range(4):
            segment = line[25*q:25*q + 26]
            # Explicit trapezoidal quadrature, independent of NumPy version.
            values.append(float((0.5*segment[0] + np.sum(segment[1:-1], dtype=np.float64)
                                 + 0.5*segment[-1]) / 25.0))
        values.extend((float(np.std(line, ddof=0)), float(np.max(line)-np.min(line))))
    return np.asarray(values, dtype=np.float64)


def independent_d(means):
    answer = []
    for line in means:
        for q in range(4):
            segment = line[25*q:25*q+26]
            answer.append(float(np.mean(0.5*(segment[:-1]+segment[1:]), dtype=np.float64)))
        centre = float(np.mean(line, dtype=np.float64))
        answer.append(float(np.sqrt(np.mean((line-centre)**2, dtype=np.float64))))
        answer.append(float(np.ptp(line)))
    return np.asarray(answer, dtype=np.float64)


def new_v_for_fixture(channels):
    return np.asarray([
        np.sqrt(np.mean(np.var(np.asarray(channels[name], dtype=np.float64),
                               axis=0, ddof=1), dtype=np.float64))
        for name in CHANNELS], dtype=np.float64)


phase = np.arange(101, dtype=np.float64)
synthetic = {name: np.stack([j+phase, j+4.0+phase])
             for j, name in enumerate(CHANNELS)}
mean_fixture = frozen_means(synthetic)
expected_means = np.stack([j+2.0+phase for j in range(9)])
check(np.allclose(mean_fixture, expected_means, atol=1e-12, rtol=0),
      'Synthetic trial-mean identity failed')
check(np.allclose(independent_means(synthetic), expected_means, atol=1e-12, rtol=0),
      'Independent trial-mean identity failed')
expected_d = np.asarray([x for j in range(9)
                         for x in ([j+2.0+25.0*q+12.5 for q in range(4)]
                                   + [np.sqrt(850.0), 100.0])], dtype=np.float64)
check(np.allclose(frozen_d(mean_fixture), expected_d, atol=1e-12, rtol=0),
      'Synthetic descriptor arithmetic failed')
check(np.allclose(independent_d(mean_fixture), expected_d, atol=1e-12, rtol=0),
      'Independent descriptor arithmetic failed')
check(np.allclose(new_v_for_fixture(synthetic), np.sqrt(8.0), atol=1e-12, rtol=0),
      'Between-cycle synthetic dispersion arithmetic failed')
print('Analytic waveform, descriptor and between-cycle arithmetic fixtures: PASS')

## Prepared-source and feature preflight


In [ ]:
# 3 — Read corrected HDF5 locally; enforce exact population and feature contracts
trial_rows, mean_rows, d_rows, index_rows = [], [], [], []
stored_unrestricted = 0
excluded = []

with h5py.File(H5, 'r') as h5:
    check('subjects' in h5, 'Missing /subjects root')
    subjects = sorted(h5['subjects'].keys())
    check(subjects == [f'S{i:02d}' for i in range(1,22)], 'Unexpected subject identities')
    check(sum(len(h5['subjects'][s]['trials']) for s in subjects) == 559,
          'Stored trial count does not match corrected source audit')
    for sid in subjects:
        trials = h5['subjects'][sid]['trials']
        for tid in sorted(trials.keys()):
            if not (tid.startswith('T') and tid[1:].isdigit()
                    and 1 <= int(tid[1:]) <= 15):
                continue
            stored_unrestricted += 1
            uid = f'{sid}_{tid}'
            group = trials[tid]
            check(all(c in group for c in CHANNELS), f'{uid}: missing angle channel')
            check('metcost_mean' in group and 'original_cycle_number' in group,
                  f'{uid}: missing target or source cycle indices')
            n = int(group[CHANNELS[0]].shape[0])
            check(n >= 20, f'{uid}: below prepared-source minimum')
            channel_arrays = {}
            for name in CHANNELS:
                data = np.asarray(group[name][()], dtype=np.float64)
                check(data.shape == (n, 101), f'{uid}: {name} shape/alignment mismatch')
                check(np.isfinite(data).all(), f'{uid}: {name} contains nonfinite values')
                channel_arrays[name] = data
            target_array = np.asarray(group['metcost_mean'][()])
            check(target_array.size == 1 and np.issubdtype(target_array.dtype, np.number),
                  f'{uid}: metabolic target must be one scalar')
            y = float(target_array.reshape(-1)[0])
            check(np.isfinite(y) and y > 0, f'{uid}: invalid corrected metabolic target')
            raw_indices = np.asarray(group['original_cycle_number'][()]).reshape(-1)
            check(len(raw_indices) == n and np.issubdtype(raw_indices.dtype, np.number),
                  f'{uid}: original-cycle index type/length invalid')
            check(np.isfinite(raw_indices).all() and np.all(raw_indices == np.floor(raw_indices)),
                  f'{uid}: original-cycle indices not finite integers')
            indices = raw_indices.astype(np.int64)
            check(np.all(np.diff(indices) > 0), f'{uid}: non-increasing original indices')
            if n < 100:
                excluded.append((uid, n))
                continue

            m = frozen_means(channel_arrays)
            m2 = independent_means(channel_arrays)
            check(np.allclose(m, m2, atol=ATOL_DEGREES, rtol=0),
                  f'{uid}: independently computed trial means disagree')
            d = frozen_d(m)
            d2 = independent_d(m)
            check(np.allclose(d, d2, atol=ATOL_DEGREES, rtol=0),
                  f'{uid}: descriptor methods disagree')
            check(m.shape == (9,101) and d.shape == (54,), f'{uid}: feature dimensions')
            tnum = int(tid[1:]); speed_idx, cadence_idx = divmod(tnum-1, 5)
            trial_rows.append(dict(trial_uid=uid, subject_id=sid, trial_id=tid,
                                   outer_fold=FOLD_OF[sid], retained_cycles=n,
                                   condition_index=tnum-1, speed_index=speed_idx,
                                   speed_m_s=SPEEDS[speed_idx],
                                   relative_cadence=CADENCES[cadence_idx],
                                   observed_W_per_kg=y,
                                   first_original_cycle_index=int(indices[0]),
                                   last_original_cycle_index=int(indices[-1]),
                                   whole_trial_missing_source_indices=int(indices[-1]-indices[0]+1-n)))
            mean_rows.append(m.reshape(909))
            d_rows.append(d)
            index_rows.extend((uid, r, int(v)) for r, v in enumerate(indices))

check(stored_unrestricted == 285, 'Unexpected stored unrestricted trial count')
check(excluded == [('S06_T11', 85)], f'Unexpected <100-cycle exclusion: {excluded}')
check(len(trial_rows) == 284 and len(set(r['trial_uid'] for r in trial_rows)) == 284,
      'Incorrect primary UID population')
check(len(set(r['subject_id'] for r in trial_rows)) == 21, 'Incorrect participant population')
check(sum(r['retained_cycles'] for r in trial_rows) == 60757, 'Incorrect retained cycle total')
check([sum(r['outer_fold']==f for r in trial_rows) for f in range(1,6)] ==
      [55,53,56,65,55], 'Incorrect original outer fold trial counts')
W = np.vstack(mean_rows).astype(np.float64)
D = np.vstack(d_rows).astype(np.float64)
check(W.shape == (284,909) and D.shape == (284,54), 'Invalid full feature matrices')
check(np.isfinite(W).all() and np.isfinite(D).all(), 'Nonfinite frozen features')
print('Local source scan complete: 284 eligible trials; 21 participants; 60,757 cycles.')

## Participant and partition reconciliation


In [ ]:
# 4 — Verify original trial ledger, exact outer and inner manifests
check(len(ref_ledger)==284 and len(ref_outer)==5 and len(ref_inner)==20,
      'Incomplete original trial/fold manifests')
ref_by_uid={r['trial_uid']: r for r in ref_ledger}
check(len(ref_by_uid)==284 and set(ref_by_uid)=={r['trial_uid'] for r in trial_rows},
      'Original trial UID set differs from corrected source')
for r in trial_rows:
    rr=ref_by_uid[r['trial_uid']]
    for key in ('subject_id','trial_id'):
        check(str(r[key])==rr[key], f'{r["trial_uid"]}: {key} mismatch')
    for key in ('outer_fold','retained_cycles','condition_index','speed_index'):
        check(int(r[key])==int(rr[key]), f'{r["trial_uid"]}: {key} mismatch')
    for key in ('speed_m_s','relative_cadence'):
        check(abs(float(r[key])-float(rr[key]))<1e-12,
              f'{r["trial_uid"]}: {key} mismatch')
    check(abs(r['observed_W_per_kg']-float(rr['observed_W_per_kg'])) <= 1e-12,
          f'{r["trial_uid"]}: original target mismatch')

all_people={r['subject_id'] for r in trial_rows}
for item in ref_outer:
    fold=int(item['outer_fold']); dev=split_people(item['development_participants'])
    test=split_people(item['test_participants'])
    check(test==set(OUTER[fold]) and dev==all_people-test and not dev.intersection(test),
          f'Outer fold {fold} participant membership mismatch')
    check(sum(r['subject_id'] in test for r in trial_rows)==int(item['n_test_trials'])
          and sum(r['subject_id'] in dev for r in trial_rows)==int(item['n_development_trials']),
          f'Outer fold {fold} trial count mismatch')
for item in ref_inner:
    fold=int(item['outer_fold']); train=split_people(item['train_participants'])
    valid=split_people(item['validation_participants']); outer_test=set(OUTER[fold])
    check(train.isdisjoint(valid) and train.isdisjoint(outer_test)
          and valid.isdisjoint(outer_test) and train|valid == all_people-outer_test,
          f'Inner fold {fold}/{item["inner_fold"]} participant leakage or omission')
    check(split_people(item['outer_test_participants'])==outer_test,
          'Inner manifest outer-test identity mismatch')
    check(sum(r['subject_id'] in train for r in trial_rows)==int(item['n_train_trials'])
          and sum(r['subject_id'] in valid for r in trial_rows)==int(item['n_validation_trials']),
          f'Inner fold {fold}/{item["inner_fold"]} trial count mismatch')
check(set((int(r['outer_fold']),int(r['inner_fold'])) for r in ref_inner)==
      {(f,k) for f in range(1,6) for k in range(1,5)},
      'Missing/duplicated original inner fold IDs')

# The original population bootstrap schedule must also remain fixed.
bootstrap_rows=csv_rows(bootstrap_blob)
check(len(bootstrap_rows)==10000, 'Original E5.2 population bootstrap draw count mismatch')
for i, row in enumerate(bootstrap_rows, start=1):
    people=split_people(row['population_sampled_participants'])
    drawn=row['population_sampled_participants'].split(';')
    check(int(row['draw_index'])==i and len(drawn)==21
          and people.issubset(all_people) and len(people)==int(row['population_distinct_participants']),
          f'Invalid original population bootstrap draw {i}')
print('Original UID/target/condition/fold manifests and bootstrap: PASS')

## Export verified feature records


In [ ]:
# 5 — Write immutable canonical preflight artifacts, provenance and PASS summary
OUT.mkdir(exist_ok=True)


def write_csv(path, header, rows):
    with path.open('w', newline='', encoding='utf-8') as f:
        w=csv.writer(f);w.writerow(header);w.writerows(rows)


def flatten_row(row, cols):
    return [row[col] for col in cols]

trial_cols=list(trial_rows[0].keys())
write_csv(OUT/'canonical_trial_ledger.csv', trial_cols,
          [flatten_row(r,trial_cols) for r in trial_rows])
wave_cols=[f'{c}_phase_{i:03d}' for c in CHANNELS for i in range(101)]
d_cols=[f'{c}_{name}' for c in CHANNELS
        for name in ('q0','q1','q2','q3','phase_sd','rom')]
write_csv(OUT/'canonical_W_ALL9.csv', ['trial_uid']+wave_cols,
          [[r['trial_uid']]+[format(v,'.17g') for v in W[j]]
           for j,r in enumerate(trial_rows)])
write_csv(OUT/'canonical_D_ALL9.csv', ['trial_uid']+d_cols,
          [[r['trial_uid']]+[format(v,'.17g') for v in D[j]]
           for j,r in enumerate(trial_rows)])
write_csv(OUT/'canonical_original_cycle_indices.csv',
          ['trial_uid','retained_row_0based','original_cycle_number'],index_rows)
with zipfile.ZipFile(REFERENCE) as z:
    for name in ('outer_fold_manifest.csv','inner_fold_manifest.csv'):
        (OUT/name).write_bytes(z.read('04/'+name))

files=sorted(p for p in OUT.iterdir() if p.is_file() and p.name != 'EXT_S0_summary.json')
checksums={p.name:sha256_file(p) for p in files}
summary=dict(status='PASS', gate='EXT-S0_LOCAL_CANONICAL_PREFLIGHT',
             approved_method_release=AMENDMENT_ID,
             source_hdf5_sha256=source_sha, reference_archive_sha256=EXPECTED_REFERENCE_SHA,
             bootstrap_population_draws_sha256=EXPECTED_BOOTSTRAP_SHA,
             primary_trials=len(trial_rows),participants=len(all_people),
             retained_cycles=sum(r['retained_cycles'] for r in trial_rows),
             frozen_full_waveform_dimension=W.shape[1],
             frozen_descriptor_dimension=D.shape[1],
             outer_fold_test_counts=[55,53,56,65,55],
             synthetic_arithmetic='PASS', original_ledger_folds_targets='PASS',
             mean_descriptor_independent_numeric_checks='PASS',
             feature_numeric_tolerance_degrees=ATOL_DEGREES,
             files_sha256=checksums,
             environment=dict(python=sys.version.split()[0],numpy=np.__version__,
                              h5py=h5py.__version__,platform=platform.platform()),
             note='Local preflight only: no E3.4 or E5.3 training/scoring; downstream cross-experiment equality and V1 gates remain PENDING.')
(OUT/'EXT_S0_summary.json').write_text(json.dumps(summary,indent=2)+'\n',encoding='utf-8')
print('EXT-S0 local canonical preflight: PASS')
print('Saved',len(files)+1,'files in',OUT)
print('The downstream E3.4-V1 and E5.3-V1 gates remain PENDING.')